# 2026-09-25 → 10-01 — Tags χρέους (52% N/A στο leverage)
**Σε εξέλιξη.**

| Βήμα | Κατάσταση |
|---|---|
| Μέτρηση κάλυψης: leverage N/A 51.9% | ✅ |
| Δείγμα 40: τα νέα tags καλύπτουν 25/40 | ✅ |
| 4 νέα tags στο `fundamentals.py` (v0.1.22 → v0.1.23) | ✅ |
| Pre-flight: 297/347 έχουν κάποιο νέο tag (86%), 235/347 και στις 2 τελευταίες χρονιές (68%) | ✅ |
| Λογική τιμών: debt/assets median 0.29· ύποπτα CMG/TDG/YUM σωστά, MAR λάθος, SNDK ανοιχτό | ✅ |
| Εγγραφή: 295 εταιρείες (χωρίς MAR, SNDK), 2.962 γρ. (30/9) + 17.608 γρ. (1/10) | ✅ |
| Σύνθεση γραμμών 30/9 (μόνο tags χρέους;) | ☐ |
| MAR / SNDK: τι πιάνει το tag | ☐ |
| Baseline → `line_items.py`/`piotroski.py` → regression + κάλυψη | ☐ |

**Αναίρεση:** `inserted_at >= '2026-09-30 08:45:00+00'` για τα tickers της εγγραφής.

**Μάθημα:** η πρώτη εγγραφή (30/9) έστελνε όλο το ιστορικό κάθε εταιρείας (~46s/εταιρεία) και κόλλησε στην 50ή σε νεκρή σύνδεση. Η τελική φιλτράρει μόνο τα νέα tags και χρησιμοποιεί σύνδεση με keepalives / pool_recycle. ~23s/εταιρεία, κυρίως κατέβασμα από SEC.

In [ ]:
# ── SETUP (dev notebooks ζουν στο dev/, αλλά δουλεύουν από τη ρίζα του repo) ──
import os, sys
if os.path.basename(os.getcwd()) == "dev":
    os.chdir("..")                      # ώστε να βρίσκει data/ και quality_scores/
sys.path.insert(0, os.getcwd())

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text
from db import get_engine
eng = get_engine()
print("Φάκελος εργασίας:", os.getcwd())
from fundamentals import fetch_companyfacts, parse_companyfacts, upsert_fundamentals


## Διάγνωση (ιστορικό — αποτελέσματα από 24–25/9)

In [ ]:
from data_quality import clean_universe

report = data_quality_report(eng)
universe = [t for t in clean_universe(report, window_years=3)
            if report.at[t, "last_year"] >= 2025]   # μόνο εταιρείες με πρόσφατα δεδομένα

rows = []
for i, t in enumerate(universe, 1):
    r = piotroski_f_score(t, eng)
    if r is not None:
        rows.append({"ticker": t, **r.breakdown})
    if i % 100 == 0:
        print(f"  ... {i}/{len(universe)}")

cov = pd.DataFrame(rows).set_index("ticker")
print(f"\n{len(cov)} εταιρείες με score από {len(universe)}")
print("\n% εταιρειών όπου κάθε κριτήριο ΔΕΝ αξιολογήθηκε:")
print((cov.isna().mean() * 100).round(1).sort_values(ascending=False).to_string())
cov.to_csv("data/piotroski_coverage.csv")

  ... 100/680
  ... 200/680
  ... 300/680
  ... 400/680
  ... 500/680
  ... 600/680

668 εταιρείες με score από 680

% εταιρειών όπου κάθε κριτήριο ΔΕΝ αξιολογήθηκε:
leverage_decreased         51.9
gross_margin_improved      40.4
current_ratio_improved     16.9
asset_turnover_improved     4.9
no_dilution                 4.5
roa_improved                1.6
positive_roa                0.1
cfo_exceeds_net_income      0.1
positive_cfo                0.0


In [ ]:
from sector_lookup import get_sectors

sectors = cov.index.map(get_sectors(list(cov.index)))
cols = ["leverage_decreased", "gross_margin_improved", "current_ratio_improved"]
by_sector = (cov[cols].isna().groupby(sectors).mean() * 100).round(0)
by_sector.insert(0, "companies", cov.groupby(sectors).size())
print(by_sector.sort_values("leverage_decreased", ascending=False).to_string())

🔄 Sector + Market Cap lookup for 5 tickers...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BK"}}}


                        companies  leverage_decreased  gross_margin_improved  current_ratio_improved
ticker                                                                                              
Real Estate                    41                88.0                   68.0                    78.0
Financial Services             90                79.0                   92.0                    79.0
Unknown                         6                67.0                   50.0                    33.0
Energy                         38                50.0                   61.0                     0.0
Consumer Cyclical              92                47.0                   27.0                     5.0
Healthcare                     76                46.0                   20.0                     0.0
Industrials                   103                45.0                   34.0                     3.0
Technology                    101                43.0                   12.0               

In [ ]:
from collections import Counter

na_debt = cov.index[cov["leverage_decreased"].isna()]
sample = list(pd.Series(na_debt).sample(min(40, len(na_debt)), random_state=1))

# Το CIK από τη βάση μας, όχι από τη λίστα της SEC (για να μη σκοντάψουμε ξανά σε BK/EQR)
cik_map = pd.read_sql(text("""
    SELECT DISTINCT ON (ticker) ticker, cik FROM fundamentals
    WHERE ticker = ANY(:t) ORDER BY ticker, filed_date DESC
"""), eng, params={"t": sample}).set_index("ticker")["cik"]

counter = Counter()
for t, cik in cik_map.items():
    us_gaap = fetch_companyfacts(int(cik)).get("facts", {}).get("us-gaap", {})
    tags = {
        tag for tag, concept in us_gaap.items()
        if any(k in tag for k in ("Debt", "Borrowing", "Notes"))
        and any(o.get("form") == "10-K" and o.get("end", "") >= "2024-01-01"
                for obs in concept.get("units", {}).values() for o in obs)
    }
    counter.update(tags)

print(f"Tags σε {len(cik_map)} εταιρείες με N/A στο χρέος:\n")
for tag, n in counter.most_common(25):
    print(f"{n:>3}/{len(cik_map)}  {tag}")

Tags σε 40 εταιρείες με N/A στο χρέος:

 23/40  ProceedsFromIssuanceOfLongTermDebt
 18/40  RepaymentsOfLongTermDebt
 17/40  LongTermDebt
 15/40  AvailableForSaleSecuritiesDebtSecurities
 14/40  AvailableForSaleDebtSecuritiesAccumulatedGrossUnrealizedLossBeforeTax
 14/40  AvailableForSaleDebtSecuritiesAccumulatedGrossUnrealizedGainBeforeTax
 14/40  LongTermDebtMaturitiesRepaymentsOfPrincipalInYearFour
 13/40  PaymentsToAcquireAvailableForSaleSecuritiesDebt
 13/40  LongTermDebtMaturitiesRepaymentsOfPrincipalInYearThree
 13/40  LongTermDebtMaturitiesRepaymentsOfPrincipalInYearFive
 13/40  LongTermDebtMaturitiesRepaymentsOfPrincipalAfterYearFive
 13/40  LongTermDebtMaturitiesRepaymentsOfPrincipalInYearTwo
 13/40  LongTermDebtMaturitiesRepaymentsOfPrincipalInNextTwelveMonths
 12/40  ProceedsFromSaleOfAvailableForSaleSecuritiesDebt
 12/40  LongTermDebtAndCapitalLeaseObligations
 11/40  AvailableForSaleDebtSecuritiesAmortizedCostBasis
 10/40  DebtSecuritiesAvailableForSaleUnrealizedLossPositi

In [ ]:
NEW_DEBT_TAGS = [
    "LongTermDebt",
    "LongTermDebtAndCapitalLeaseObligationsIncludingCurrentMaturities",
    "LongTermDebtAndCapitalLeaseObligations",
]

def annual_ends(concept):
    """Ημερομηνίες λήξης χρήσης για τις οποίες το tag έχει τιμή σε κάποιο 10-K."""
    return {o["end"] for obs in concept.get("units", {}).values() for o in obs
            if o.get("form") == "10-K"}

covered = {}
for t, cik in cik_map.items():
    us_gaap = fetch_companyfacts(int(cik)).get("facts", {}).get("us-gaap", {})
    last_two = sorted(annual_ends(us_gaap.get("Assets", {})))[-2:]   # φέτος και πέρσι
    covered[t] = next((tag for tag in NEW_DEBT_TAGS
                       if tag in us_gaap and set(last_two) <= annual_ends(us_gaap[tag])), None)

res = pd.Series(covered)
print(f"Θα αξιολογούνταν στο χρέος: {res.notna().sum()} από {len(res)}\n")
print(res.value_counts().to_string())
print("\nΑκόμα χωρίς:", sorted(res[res.isna()].index))

Θα αξιολογούνταν στο χρέος: 25 από 40

LongTermDebt                                                        17
LongTermDebtAndCapitalLeaseObligationsIncludingCurrentMaturities     7
LongTermDebtAndCapitalLeaseObligations                               1

Ακόμα χωρίς: ['ACGL', 'BXP', 'CCB', 'CDNS', 'DDOG', 'ERIE', 'ESS', 'EXR', 'ILMN', 'MMI', 'PLTR', 'TTD', 'VRSN', 'WRB', 'XYZ']


## Pre-flight — μόνο ανάγνωση (~5–15 λεπτά)

In [ ]:
import importlib.metadata as m
import pandas as pd
from sqlalchemy import text
from fundamentals import parse_companyfacts, fetch_companyfacts, XBRL_TAG_MAP

# ── 0. Έλεγχος: βλέπει το notebook τη νέα έκδοση; ─────────────────────────
NEW_DEBT = [
    "LongTermDebt",
    "LongTermDebtAndCapitalLeaseObligations",
    "LongTermDebtAndCapitalLeaseObligationsCurrent",
    "LongTermDebtAndCapitalLeaseObligationsIncludingCurrentMaturities",
]
print("Έκδοση πακέτου:", m.version("trading-shared-data"))
missing = [t for t in NEW_DEBT if t not in XBRL_TAG_MAP]
assert not missing, f"ΣΤΑΜΑΤΑ: το notebook βλέπει παλιό fundamentals.py (λείπουν {missing})"
print("Τα 4 νέα tags υπάρχουν ✓\n")

# ── 1. Ποιες εταιρείες ελέγχουμε ───────────────────────────────────────────
cov = pd.read_csv("data/piotroski_coverage.csv", index_col=0)
targets = cov.index[cov["leverage_decreased"].isna()].tolist()
print(f"{len(targets)} tickers με N/A στο χρέος")

# CIK από τη δική μας βάση (όχι από τη λίστα της SEC, για να μη χαθούν BK κ.λπ.)
cik_map = pd.read_sql(text("""
    SELECT DISTINCT ON (ticker) ticker, cik FROM fundamentals
    WHERE ticker = ANY(:t) ORDER BY ticker, filed_date DESC
"""), eng, params={"t": targets}).set_index("ticker")["cik"]
no_cik = sorted(set(targets) - set(cik_map.index))
print(f"Με CIK: {len(cik_map)}   Χωρίς CIK: {no_cik}\n")

# ── 2. Κατέβασμα + parse, ΧΩΡΙΣ εγγραφή ───────────────────────────────────
previews, gained, failed = {}, [], {}
for i, (t, cik) in enumerate(cik_map.items(), 1):
    try:
        df = parse_companyfacts(t, int(cik), fetch_companyfacts(int(cik)))
    except Exception as e:
        failed[t] = str(e)[:80]
        continue
    previews[t] = df
    if df["metric"].isin(NEW_DEBT).any():
        gained.append(t)
    if i % 50 == 0:
        print(f"  ... {i}/{len(cik_map)}")

# ── 3. Αποτέλεσμα ─────────────────────────────────────────────────────────
print(f"\nΘα αποκτήσουν δεδομένα χρέους: {len(gained)} / {len(cik_map)} "
      f"({len(gained) / len(cik_map):.0%})")
print("Αποτυχίες λήψης:", failed or "καμία")

by_tag = pd.Series({t: next(x for x in NEW_DEBT if x in set(previews[t]["metric"]))
                    for t in gained})
print("\nΠρώτο διαθέσιμο νέο tag ανά εταιρεία:")
print(by_tag.value_counts().to_string())

Έκδοση πακέτου: 0.1.23
Τα 4 νέα tags υπάρχουν ✓

347 tickers με N/A στο χρέος
Με CIK: 347   Χωρίς CIK: []



## Εγγραφή — ΜΟΝΟ μετά από έγκριση του pre-flight

In [ ]:
# ⛔ ΕΤΡΕΞΕ 2026-10-01: 295 εταιρείες, 17.608 νέες γραμμές, 8.158s, 0 αποτυχίες
RUN_WRITE = False
assert RUN_WRITE, "Κελί εγγραφής απενεργοποιημένο (δες RUN_WRITE)"

import time
from sqlalchemy import create_engine
from db import get_database_url

# Σύνδεση που αντέχει μεγάλες αδράνειες (η get_engine() κόλλησε σε νεκρή σύνδεση στις 30/9)
eng = create_engine(
    get_database_url(), pool_pre_ping=True, pool_recycle=300,
    connect_args={"connect_timeout": 10, "keepalives": 1, "keepalives_idle": 30,
                  "keepalives_interval": 10, "keepalives_count": 3},
)

NEW_DEBT = ["LongTermDebt", "LongTermDebtAndCapitalLeaseObligations",
            "LongTermDebtAndCapitalLeaseObligationsCurrent",
            "LongTermDebtAndCapitalLeaseObligationsIncludingCurrentMaturities"]
EXCLUDE = {"MAR", "SNDK"}
UNDO_FROM = "2026-09-30 08:45:00+00"

cov = pd.read_csv("data/piotroski_coverage.csv", index_col=0)
targets = [t for t in cov.index[cov["leverage_decreased"].isna()] if t not in EXCLUDE]
cik_map = pd.read_sql(text("""
    SELECT DISTINCT ON (ticker) ticker, cik FROM fundamentals
    WHERE ticker = ANY(:t) ORDER BY ticker, filed_date DESC
"""), eng, params={"t": targets}).set_index("ticker")["cik"]

written, total_new, failed, t0 = [], 0, {}, time.time()
for i, (t, cik) in enumerate(cik_map.items(), 1):
    try:
        df = parse_companyfacts(t, int(cik), fetch_companyfacts(int(cik)))
        df = df[df["metric"].isin(NEW_DEBT)]          # ΜΟΝΟ τα νέα tags
        if not df.empty:
            total_new += upsert_fundamentals(df, eng)["inserted"]
            written.append(t)
    except Exception as e:
        failed[t] = str(e)[:80]
    if i % 25 == 0:
        print(f"  ... {i}/{len(cik_map)}  ({time.time() - t0:.0f}s, {total_new} νέες γραμμές)")
print(f"\nΓράφτηκαν {len(written)} εταιρείες | {total_new} νέες γραμμές | {time.time() - t0:.0f}s")


## Αναίρεση — μόνο αν κάτι πήγε στραβά

In [ ]:
RUN_UNDO = False
assert RUN_UNDO, "Κελί αναίρεσης απενεργοποιημένο (δες RUN_UNDO)"

UNDO_FROM = "2026-09-30 08:45:00+00"
NEW_DEBT = ["LongTermDebt", "LongTermDebtAndCapitalLeaseObligations",
            "LongTermDebtAndCapitalLeaseObligationsCurrent",
            "LongTermDebtAndCapitalLeaseObligationsIncludingCurrentMaturities"]

# Σβήνει ΜΟΝΟ γραμμές των 4 νέων tags που μπήκαν από τότε — όχι ό,τι άλλο
# έγραψε στο μεταξύ το εβδομαδιαίο ingest.
with eng.begin() as conn:
    n = conn.execute(text("""
        DELETE FROM fundamentals f USING metric_lookup m
        WHERE m.id = f.metric_id AND m.name = ANY(:tags) AND f.inserted_at >= :ts
    """), {"tags": NEW_DEBT, "ts": UNDO_FROM}).rowcount
print(f"Αναιρέθηκαν {n} γραμμές.")


## Επόμενο: αλλαγή στο `line_items.py`, μετά `dev/diagnostics.ipynb` (regression + κάλυψη)